# Optional Lab 8B — The Same Team, Three Ways

Chapter 8 built a three-agent team with a `for` loop. The book's closing claim is that
every framework you will meet rearranges these same parts and gives them new names.
This lab tests that claim on two frameworks the repository already pins and, until
now, never used: **LangGraph** and **Google ADK 2.x**.

The rule of the lab: the *team* is imported from Chapter 8's folder and does not
change. No worker, no tool, no authorization check is rewritten. Each framework is
allowed to decide exactly two things — **who runs next** and **what state carries
between them** — and is then asked for the same verdict, the same handoffs, the same
audit trail, and the same single trace id.

If a framework changes any of those, it changed the answer, and the claim is false.


## Setup

Every lab in this book installs from **one** `requirements.txt` in the companion
repository. The clone below fails loudly on purpose.


In [ ]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


In [ ]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt


Verify the environment and confirm this lab's source folder is in the checkout.


In [ ]:
!python tools/check_env.py --chapter 8b


In [ ]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key
print("model tier:", os.environ["AEGIS_MODEL"])


## Way 1 — scratch

Chapter 8's orchestrator, verbatim. Four lines. Keep the shape in mind: a loop over
workers, an envelope passed along.


In [ ]:
import sys, inspect
sys.path.insert(0, "labs/chapter-08b-the-same-team-three-ways")

from frameworks.team import soc
from frameworks import scratch_team
from frameworks.scratch_team import run_scratch

print(inspect.getsource(run_scratch))
scratch = run_scratch(soc.SEED_ALERT, trace_id="inc-8b")
print("verdict:", scratch.verdict, "| severity:", scratch.severity, "| ticket:", scratch.ticket_id)
print("hops:   ", scratch.hops)


## Way 2 — LangGraph

A `StateGraph` whose **state is the envelope**: the fields of Chapter 8's `A2AMessage`
as a `TypedDict`. Each node rebuilds the message from state, calls the Chapter 8
worker, and writes the returned envelope back. The edges are the orchestrator.


In [ ]:
from frameworks.langgraph_team import build_graph, run_langgraph

graph = build_graph()
g = graph.get_graph()
print("nodes:", [n for n in g.nodes if n not in ("__start__", "__end__")])
print("edges:", [f"{e.source} -> {e.target}" for e in g.edges])
print()

lg = run_langgraph(soc.SEED_ALERT, trace_id="inc-8b")
print("verdict:", lg.verdict, "| severity:", lg.severity, "| ticket:", lg.ticket_id)
print("hops:   ", lg.hops)


### What LangGraph adds: a checkpoint after every node

Run the same graph under a `MemorySaver` and every node boundary becomes a saved
state. That is the feature the loop cannot give you: inspect a run step by step,
or resume it from the node that failed instead of starting over. (Chapter 12
called this durable execution; this is the primitive.)


In [ ]:
from frameworks.langgraph_team import checkpoint_history

for snap in checkpoint_history(soc.SEED_ALERT, "inc-8b"):
    print(f'  step {snap["step"]}  next={str(snap["next"]):20} envelope to={str(snap["to_agent"]):14} payload keys={snap["keys"]}')
print()
print("Each line is a resumable state. A crash after 'investigate' restarts at 'report'.")


## Way 3 — Google ADK 2.x

A `Workflow` whose **nodes are the workers**: `@node` wraps a plain async function, the
`edges` list from `START` defines the order, and the session state carries the
envelope. The runner is in-memory, so this runs offline exactly like the mock tier.

Note the version. ADK 2.x replaced `SequentialAgent` with `Workflow` and `LlmAgent`
with `Agent` — the same breaking change the book's `requirements.txt` warns about.


In [ ]:
from frameworks.adk_team import build_workflow, run_adk

wf = build_workflow()
print("workflow:", wf.name)
print("edges:   ", [f"{getattr(a, 'name', a)} -> {getattr(b, 'name', b)}" for a, b in wf.edges])
print()

adk = run_adk(soc.SEED_ALERT, trace_id="inc-8b")
print("verdict:", adk.verdict, "| severity:", adk.severity, "| ticket:", adk.ticket_id)
print("hops:   ", adk.hops)


### What ADK adds: a scheduler that runs independent nodes concurrently

Chapter 8's fan-out used a thread pool by hand. In ADK the graph *is* the schedule:
two branches from `START`, a `JoinNode` that waits for both, and a merge that reports
dissent — the same lesson as §8.4, with no thread code.


In [ ]:
from frameworks.adk_team import fan_out_workflow

def investigate_signal(signal: str) -> dict:
    if "auth" in signal or "privilege" in signal:
        return {"verdict": "confirmed_compromise", "severity": "critical"}
    return {"verdict": "inconclusive", "severity": "low"}

merged = fan_out_workflow(["failed auth burst", "unusual source ip", "privilege escalation"], investigate_signal)
print("merged:", merged)
print()
print("Same scatter-gather as Chapter 8. The dissent is still reported, not averaged.")


## The comparison

Three orchestrators, one team. The claim is that the mechanism does not change the
answer — so check every field that would reveal it if it did.


In [ ]:
runs = [scratch, lg, adk]

print(f'{"framework":10} {"verdict":22} {"severity":9} {"ticket":9} {"trace":8} {"orchestration lines":>20}')
for r in runs:
    print(f'{r.framework:10} {r.verdict:22} {r.severity:9} {r.ticket_id:9} {r.trace_id:8} {r.orchestration_lines:>20}')

print()
print("same verdict:     ", len({r.verdict for r in runs}) == 1)
print("same handoffs:    ", len({tuple(r.hops) for r in runs}) == 1, "->", runs[0].hops)
print("same audit trail: ", len({tuple(r.audit) for r in runs}) == 1)
print("one trace id:     ", {r.trace_id for r in runs})
writers = {agent for agent, tool, ok in runs[0].audit if tool == "create_ticket" and ok}
denied = [(agent, tool) for agent, tool, ok in runs[0].audit if not ok]
print("who touched the world:", writers, "| denied:", denied)


---

## What you built

The Chapter 8 team run by a loop, by LangGraph, and by ADK, producing the same
verdict, the same handoffs, the same audit trail, and the same trace id.

- **The mechanism did not change the answer.** It changed what you get for free:
  LangGraph gives you a checkpoint at every node; ADK gives you a scheduler with
  retries, timeouts and concurrency per node.
- **The security properties survived every framework** because they never lived in
  the orchestrator. Least privilege is a list inside the worker; the audit is written
  by the authorization check; neither framework was consulted.
- **Choose by what you need to survive**, not by novelty. If runs must resume after a
  crash, take the checkpointer. If nodes must retry with backoff and fan out, take the
  scheduler. If neither, the loop is four lines and you can read all of them.

The parts the book taught are the parts both frameworks are made of. That was the claim.
